In [ ]:
library(CellChat)
library(Seurat)
library(NMF)
library(ggalluvial)
library(ComplexHeatmap)
library(dplyr)
library(ggplot2)
library(ggrepel)
library(circlize)
library(igraph)
library(RColorBrewer)
library(scales)

options(stringsAsFactors = FALSE)


In [ ]:
# Project configuration.
outputdir <- file.path("transcriptome", "pool_DEG")
dir.create(outputdir, recursive = TRUE, showWarnings = FALSE)

# By default use the curated mouse database bundled with CellChat.
# To use CellPhoneDB-derived tables, set use_custom_cellphonedb to TRUE and
# provide CELLPHONEDB_DIR or edit cellphoneDB_dir below.
use_custom_cellphonedb <- FALSE
cellphoneDB_dir <- Sys.getenv("CELLPHONEDB_DIR", unset = "")


In [ ]:
if (use_custom_cellphonedb) {
  if (!nzchar(cellphoneDB_dir)) {
    stop("Set CELLPHONEDB_DIR before using the custom CellPhoneDB-derived database.")
  }

  db_files <- c(
    interaction = "interaction_input_CellChatDB.csv",
    complex = "complex_input_CellChatDB.csv",
    cofactor = "cofactor_input_CellChatDB.csv",
    geneInfo = "geneInfo_input_CellChatDB.csv"
  )
  db_paths <- file.path(cellphoneDB_dir, db_files)
  names(db_paths) <- names(db_files)
  missing_db_files <- db_paths[!file.exists(db_paths)]
  if (length(missing_db_files) > 0) {
    stop("Missing custom database files: ", paste(missing_db_files, collapse = ", "))
  }

  CellChatDB <- lapply(db_paths, read.csv, row.names = 1, check.names = FALSE)
} else {
  CellChatDB <- CellChatDB.mouse
}


In [ ]:
required_db_tables <- c("interaction", "complex", "cofactor", "geneInfo")
missing_db_tables <- setdiff(required_db_tables, names(CellChatDB))
if (length(missing_db_tables) > 0) {
  stop("CellChat database is missing: ", paste(missing_db_tables, collapse = ", "))
}
message("Using ", if (use_custom_cellphonedb) "custom CellPhoneDB-derived" else "CellChatDB.mouse", " database.")


In [ ]:
project_list=c('M20_3_V4', 'M20_3_V5', 'M20_5_V21',
       'M20_5_V47', 'M20_V40', 'M20_V42')

In [ ]:
# All inputs and outputs below are relative to this directory.
outputdir


In [ ]:
counts_file <- file.path(outputdir, "pool_raw.csv")
if (!file.exists(counts_file)) stop("Counts file not found: ", counts_file)
counts <- read.csv(counts_file, row.names = 1, check.names = FALSE)


In [ ]:
metadata_file <- file.path(outputdir, "meta_treatment_anno.csv")
if (!file.exists(metadata_file)) stop("Metadata file not found: ", metadata_file)
meta_everything_anno <- read.csv(metadata_file, row.names = 1, check.names = FALSE)


In [ ]:
required_metadata <- c("anno_sub", "Treatment_Sample")
missing_metadata <- setdiff(required_metadata, colnames(meta_everything_anno))
if (length(missing_metadata) > 0) {
  stop("Missing metadata columns: ", paste(missing_metadata, collapse = ", "))
}
if (!setequal(colnames(counts), rownames(meta_everything_anno))) {
  stop("Counts columns and metadata row names must contain the same cell barcodes.")
}
meta_everything_anno <- meta_everything_anno[colnames(counts), , drop = FALSE]

# Use one macrophage label consistently across the Python, Monocle and CellChat notebooks.
meta_everything_anno$anno_sub[meta_everything_anno$anno_sub == "M2.TAM"] <- "TAM.M2"
meta_everything_anno$anno_sub <- factor(meta_everything_anno$anno_sub)
meta_everything_anno$Treatment_Sample <- factor(meta_everything_anno$Treatment_Sample)

data <- CreateSeuratObject(counts = counts, project = "all", meta.data = meta_everything_anno)
data <- NormalizeData(data, scale.factor = 1e6, normalization.method = "LogNormalize")


In [ ]:
data$anno_sub <- droplevels(factor(data$anno_sub))
data$Treatment_Sample <- droplevels(factor(data$Treatment_Sample))


In [ ]:
table(data$Treatment_Sample)
table(data$anno_sub)


In [ ]:
for (project in project_list) {
  data_sub <- subset(data, subset = Treatment_Sample == project)
  if (ncol(data_sub) == 0) {
    warning("No cells found for sample ", project, "; skipping.")
    next
  }

  data_sub <- FindVariableFeatures(data_sub, selection.method = "vst", nfeatures = 20000)
  data_sub <- ScaleData(data_sub)
  cellchat <- createCellChat(object = data_sub, group.by = "anno_sub")
  cellchat <- setIdent(cellchat, ident.use = "anno_sub")
  cellchat@DB <- CellChatDB
  cellchat <- subsetData(cellchat)

  cellchat <- identifyOverExpressedGenes(cellchat, thresh.pc = 0.1, thresh.fc = 2, thresh.p = 0.05)
  cellchat <- identifyOverExpressedInteractions(cellchat)
  cellchat@idents <- droplevels(cellchat@idents)

  k_min <- max(5L, as.integer(floor(sqrt(ncol(data_sub)))))
  min_cells <- max(10L, as.integer(floor(2 * sqrt(ncol(data_sub)))))
  cellchat <- computeCommunProb(
    cellchat,
    raw.use = TRUE,
    type = "truncatedMean",
    k.min = k_min,
    nboot = 24,
    population.size = TRUE
  )
  cellchat <- filterCommunication(cellchat, min.cells = min_cells)
  cellchat <- computeCommunProbPathway(cellchat)
  cellchat <- aggregateNet(cellchat)

  groupSize <- as.numeric(table(cellchat@idents))
  pdf(file.path(outputdir, paste0(project, "_Visual_circle.pdf")), height = 10, width = 12)
  netVisual_circle(
    cellchat@net$weight,
    vertex.weight = groupSize,
    weight.scale = TRUE,
    label.edge = FALSE,
    title.name = "Interaction weights/strength"
  )
  dev.off()

  saveRDS(cellchat, file = file.path(outputdir, paste0(project, "_cellchat.rds")))
}


In [ ]:
treatment_names <- project_list
cellchats <- list()

for (treatment in treatment_names) {
  filename <- file.path(outputdir, paste0(treatment, "_cellchat.rds"))
  if (!file.exists(filename)) stop("CellChat object not found: ", filename)
  message("Loading ", filename)
  cellchats[[treatment]] <- readRDS(filename)
}

message("Loaded ", length(cellchats), " CellChat objects: ", paste(names(cellchats), collapse = ", "))


In [ ]:
# 定义参考组优先级
reference_groups <- c( "M20_5_V21","M20_5_V47", "M20_3_V4","M20_3_V5")  # 优先级顺序

sample_pairs <- combn(names(cellchats), 2, simplify = FALSE)

sample_pairs <- lapply(sample_pairs, function(pair) {
  # 找到这一对中优先级最高的参考组
  for (ref in reference_groups) {
    if (ref %in% pair) {
      if (pair[1] != ref) {
        return(c(ref, setdiff(pair, ref)))
      }
      break
    }
  }
  return(pair)  # 如果没有参考组，保持原顺序
})

cat("处理顺序（按参考组优先级）:\n")
for (i in 1:length(sample_pairs)) {
  cat(i, ":", sample_pairs[[i]][1], "vs", sample_pairs[[i]][2], "\n")
}

In [ ]:
# CellChat 2×2 设计分析工具
# 完整重构版本


# ============================================================================
# 1. 数据提取函数
# ============================================================================

extract_communication_matrix <- function(cellchat_obj, sample_name, target_types) {
  
  # 检查可用细胞类型
  available_celltypes <- intersect(rownames(cellchat_obj@net$weight), target_types)
  
  # 打印信息
  cat("样本", sample_name, "中找到的细胞亚群:", 
      length(available_celltypes), "/", length(target_types), "\n")
  
  if (length(available_celltypes) > 0) {
    cat("具体亚群:", paste(available_celltypes, collapse = ", "), "\n")
  }
  
  # 创建完整矩阵（包含所有目标类型）
  full_mat <- matrix(0, 
                     nrow = length(target_types), 
                     ncol = length(target_types),
                     dimnames = list(target_types, target_types))
  
  # 填充可用数据
  if (length(available_celltypes) > 0) {
    sub_mat <- cellchat_obj@net$weight[available_celltypes, available_celltypes]
    full_mat[available_celltypes, available_celltypes] <- as.matrix(sub_mat)
  }
  
  # 转换为数据框
  df <- as.data.frame(as.table(full_mat))
  colnames(df) <- c("source", "target", "strength")
  df$sample <- sample_name
  df$available <- ifelse(df$source %in% available_celltypes & 
                         df$target %in% available_celltypes, "Yes", "No")
  
  return(df)
}

# ============================================================================
# 2. 样本比较函数
# ============================================================================

compare_two_samples <- function(df, sample1, sample2) {
  
  # 筛选两个样本的数据
  data1 <- df %>% filter(sample == sample1)
  data2 <- df %>% filter(sample == sample2)
  
  # 合并数据
  merged <- merge(data1, data2, 
                  by = c("source", "target"), 
                  suffixes = c(paste0("_", sample1), paste0("_", sample2)))
  
  # 计算差异
  strength_col1 <- paste0("strength_", sample1)
  strength_col2 <- paste0("strength_", sample2)
  
  merged$strength_diff <- merged[[strength_col1]] - merged[[strength_col2]]
  
  # 计算倍数变化（注意处理0值）
  merged$fold_change <- ifelse(merged[[strength_col1]] > 0,
                               merged[[strength_col2]] / merged[[strength_col1]],
                               NA)
  
  # 判断变化方向
  merged$change_direction <- case_when(
    merged$strength_diff > 0 ~ "upregulated",
    merged$strength_diff < 0 ~ "downregulated", 
    TRUE ~ "unchanged"
  )
  
  # 仅保留在两个样本中都可用的交互
  avail_col1 <- paste0("available_", sample1)
  avail_col2 <- paste0("available_", sample2)
  
  merged <- merged %>% 
    filter(get(avail_col1) == "Yes" & get(avail_col2) == "Yes")
  
  return(merged)
}

# ============================================================================
# 3. 小样本统计分析函数（修复版本）
# ============================================================================

#' 小样本量统计分析（2×2设计） - 修复版本
perform_statistical_analysis_2x2 <- function(all_pairwise_results, 
                                             group_A_samples, 
                                             group_B_samples) {
  
  # 验证样本数量
  expected_pairs <- length(group_A_samples) * length(group_B_samples)
  actual_pairs <- length(unique(all_pairwise_results$pair_id))
  
  if (actual_pairs != expected_pairs) {
    warning("Expected " , expected_pairs, " pairwise contrasts but found " , actual_pairs, ".")
  }
  cat("统计设计: 每组2个样本，共有", actual_pairs, "个比较对\n")
  warning("Exploratory analysis only: pairwise contrasts share samples and are not independent biological replicates.")
  
  # 第一步：计算基本统计
  statistical_results <- all_pairwise_results %>%
    group_by(source, target) %>%
    summarize(
      # 基本描述统计
      n_pairs = n(),
      mean_diff = mean(strength_diff, na.rm = TRUE),
      median_diff = median(strength_diff, na.rm = TRUE),
      sd_diff = sd(strength_diff, na.rm = TRUE),
      se_diff = ifelse(n_pairs > 1, sd_diff / sqrt(n_pairs), NA),
      
      # 方向一致性
      n_up = sum(change_direction == "upregulated"),
      n_down = sum(change_direction == "downregulated"),
      n_unchanged = sum(change_direction == "unchanged"),
      prop_same_direction = max(n_up, n_down) / n_pairs,
      dominant_direction = case_when(
        n_up == n_pairs ~ "all_up",
        n_down == n_pairs ~ "all_down",
        n_up > n_down ~ "mostly_up",
        n_down > n_up ~ "mostly_down",
        TRUE ~ "inconsistent"
      ),
      
      # 倍数变化
      median_fc = median(fold_change, na.rm = TRUE),
      mean_fc = mean(fold_change, na.rm = TRUE),
      
      .groups = "drop"
    )
  
  # 第二步：对每一行单独计算统计检验
  add_statistical_tests <- function(stat_df, pairwise_df) {
    results <- list()
    
    for (i in 1:nrow(stat_df)) {
      row <- stat_df[i, ]
      
      # 提取差异值
      diff_values <- pairwise_df %>%
        filter(source == row$source, target == row$target) %>%
        pull(strength_diff)
      
      # 初始化统计结果
      stats <- list(
        t_test_p_value = NA_real_,
        t_statistic = NA_real_,
        wilcox_p_value = NA_real_,
        sign_test_p_value = NA_real_,
        n_positive = sum(diff_values > 0, na.rm = TRUE),
        n_negative = sum(diff_values < 0, na.rm = TRUE)
      )
      
      # t检验
      if (row$n_pairs >= 2 && row$sd_diff > 0 && length(diff_values) >= 2) {
        tryCatch({
          t_result <- t.test(diff_values, mu = 0)
          stats$t_test_p_value <- t_result$p.value
          stats$t_statistic <- t_result$statistic
        }, error = function(e) {})
      }
      
      # Wilcoxon检验
      if (row$n_pairs >= 2 && length(diff_values) >= 2) {
        tryCatch({
          wilcox_result <- wilcox.test(diff_values, mu = 0, exact = FALSE)
          stats$wilcox_p_value <- wilcox_result$p.value
        }, error = function(e) {})
      }
      
      # 符号检验
      if (row$n_pairs >= 1) {
        tryCatch({
          sign_test <- binom.test(max(stats$n_positive, stats$n_negative), 
                                 row$n_pairs, p = 0.5, alternative = "two.sided")
          stats$sign_test_p_value <- sign_test$p.value
        }, error = function(e) {})
      }
      
      results[[i]] <- as.data.frame(stats)
    }
    
    # 合并结果
    tests_df <- bind_rows(results)
    return(cbind(stat_df, tests_df))
  }
  
  # 添加统计检验
  statistical_results <- add_statistical_tests(statistical_results, all_pairwise_results)
  
  # 第三步：计算衍生统计
  statistical_results <- statistical_results %>%
    mutate(
      # 选择p值（优先Wilcoxon）
      p_value_raw = case_when(
          !is.na(wilcox_p_value) ~ wilcox_p_value,
        !is.na(t_test_p_value) ~ t_test_p_value,
        
        !is.na(sign_test_p_value) ~ sign_test_p_value,
        TRUE ~ NA_real_
      ),
      
      # 效应大小
      cohens_d = ifelse(sd_diff > 0, mean_diff / sd_diff, NA_real_),
      
      # 一致性评分
      consistency_score = case_when(
        prop_same_direction == 1 ~ 4,
        prop_same_direction >= 0.75 ~ 3,
        prop_same_direction >= 0.5 ~ 2,
        TRUE ~ 1
      ),
      
      # 变异系数
      cv = ifelse(mean_diff != 0, sd_diff / abs(mean_diff), NA_real_)
    )
  
  # 第四步：多重检验校正 - 修复p.adjust的使用
  # 创建p值的副本，用于校正
  p_values <- statistical_results$p_value_raw
  
  # Always create the adjusted-p-value column, including zero/one-test edge cases.
  p_adjusted <- rep(NA_real_, length(p_values))
  non_na_indices <- which(!is.na(p_values))
  if (length(non_na_indices) > 0) {
    p_adjusted[non_na_indices] <- p.adjust(p_values[non_na_indices], method = "fdr")
  }
  statistical_results$p_adjusted <- p_adjusted
  
  # 第五步：标注显著性
  statistical_results <- statistical_results %>%
    mutate(
      significance = case_when(
        p_adjusted < 0.05 ~ "**",
        p_adjusted < 0.1 ~ "*",
        p_adjusted < 0.2 ~ ".",
        TRUE ~ "ns"
      ),
      
      priority_level = case_when(
        consistency_score == 4 & abs(mean_diff) > 0.2 ~ "high",
        consistency_score >= 3 & abs(mean_diff) > 0.15 ~ "medium_high",
        consistency_score >= 2 & abs(mean_diff) > 0.1 ~ "medium",
        TRUE ~ "low"
      )
    )
  
  # 打印摘要
  cat("\n统计分析摘要（2×2设计）:\n")
  cat("总交互作用数量:", nrow(statistical_results), "\n")
  cat("高度一致(4/4同向):", sum(statistical_results$consistency_score == 4), "\n")
  cat("中等一致(3/4同向):", sum(statistical_results$consistency_score == 3), "\n")
  cat("边缘一致(2/4同向):", sum(statistical_results$consistency_score == 2), "\n")
  cat("显著(p<0.05):", sum(statistical_results$p_adjusted < 0.05, na.rm = TRUE), "\n")
  cat("边缘显著(p<0.1):", sum(statistical_results$p_adjusted < 0.1, na.rm = TRUE), "\n")
  
  return(statistical_results)
}

# ============================================================================
# 4. 网络可视化函数
# ============================================================================

plot_cell_network_2x2 <- function(statistical_results, comp_name, outputdir) {
  
  # 筛选重要交互
  network_data <- statistical_results %>%
    filter(priority_level %in% c("high", "medium_high", "medium")) %>%
    mutate(
      edge_weight = consistency_score * abs(mean_diff),
      edge_color = case_when(
        mean_diff > 0 ~ "#E76F51",   # A>B (上调)
        mean_diff < 0 ~ "#2A9D8F",   # A<B (下调)
        TRUE ~ "gray"
      ),
      edge_width = case_when(
        priority_level == "high" ~ 4,
        priority_level == "medium_high" ~ 3,
        priority_level == "medium" ~ 2,
        TRUE ~ 1
      ),
      edge_label = paste0(
        "一致性:", consistency_score, "/4\n",
        "Δ=", round(mean_diff, 3), "\n",
        ifelse(!is.na(p_adjusted), 
               paste0("p=", round(p_adjusted, 3)), 
               "p=NA")
      )
    )
  
  # 检查是否有足够数据
  if (nrow(network_data) == 0) {
    cat("警告：没有足够一致的变化，无法绘制网络图\n")
    cat("考虑降低筛选阈值或检查数据\n")
    return(NULL)
  }
  
  # 计算节点重要性
  node_importance <- bind_rows(
    network_data %>% 
      group_by(source) %>%
      summarise(
        outgoing_strength = sum(consistency_score * abs(mean_diff), na.rm = TRUE),
        outgoing_n = n(),
        .groups = "drop"
      ) %>%
      rename(cell = source),
    network_data %>% 
      group_by(target) %>%
      summarise(
        incoming_strength = sum(consistency_score * abs(mean_diff), na.rm = TRUE),
        incoming_n = n(),
        .groups = "drop"
      ) %>%
      rename(cell = target)
  ) %>%
    group_by(cell) %>%
    summarise(
      total_strength = sum(outgoing_strength, incoming_strength, na.rm = TRUE),
      total_connections = sum(outgoing_n, incoming_n, na.rm = TRUE),
      .groups = "drop"
    )
  
  # 创建igraph对象
  g <- graph_from_data_frame(
    d = network_data[, c("source", "target", "edge_weight", 
                         "edge_color", "edge_width", "edge_label")],
    vertices = node_importance,
    directed = TRUE
  )
  
  # 设置节点属性
  V(g)$size <- scales::rescale(V(g)$total_strength, to = c(8, 25))
  V(g)$label.cex <- scales::rescale(V(g)$total_strength, to = c(0.6, 1.0))
  V(g)$color <- "lightblue"
  V(g)$frame.color <- "darkblue"
  
  # 设置边属性
  E(g)$color <- E(g)$edge_color
  E(g)$width <- E(g)$edge_width
  E(g)$arrow.size <- 0.6
  E(g)$arrow.width <- 1.0
  E(g)$label <- E(g)$edge_label
  E(g)$label.color <- "black"
  E(g)$label.cex <- 0.5
  
  # 创建输出目录
  if (!dir.exists(outputdir)) {
    dir.create(outputdir, recursive = TRUE)
  }
  
  # 保存网络图
  pdf_file <- file.path(outputdir, paste0("network_2x2_", comp_name, ".pdf"))
  pdf(pdf_file, width = 12, height = 10)
  
  par(mar = c(0, 0, 2, 0), cex.main = 1.0)
  
  plot(g,
       layout = layout_with_fr(g),
       vertex.label = V(g)$name,
       main = paste("Cell-Cell Communication Network (2×2 design):", comp_name),
       sub = paste("基于一致性评分 |", nrow(network_data), "个交互作用"))
  
  # 添加图例
  legend("bottomleft",
         legend = c("A > B (上调)", "A < B (下调)", 
                   "高优先级(4/4一致)", "中高优先级(3/4)", "中优先级(2/4)"),
         col = c("#E76F51", "#2A9D8F", "black", "black", "black"),
         lwd = c(3, 3, 4, 3, 2),
         lty = 1,
         bty = "n",
         cex = 0.7)
  
  # 添加样本信息
  legend("bottomright",
         legend = c("注意：每组只有2个样本", "统计功效有限", "结果需谨慎解释"),
         bty = "n",
         cex = 0.6,
         text.col = "red")
  
  dev.off()
  cat("网络图已保存至:", pdf_file, "\n")
  
  # 保存详细结果
  csv_file <- file.path(outputdir, paste0("statistics_2x2_", comp_name, ".csv"))
  write.csv(statistical_results, csv_file, row.names = FALSE)
  
  return(list(
    graph = g,
    network_data = network_data,
    statistical_results = statistical_results
  ))
}

# ============================================================================
# 5. 环形图可视化函数
# ============================================================================

create_directional_circos <- function(statistical_results, comp_name, outputdir, n_top = 20) {
  
  # 筛选top交互
   top_data <- statistical_results %>%
      filter(p_value_raw < 0.05) %>%  # 首先筛选显著的结果
      arrange(desc(abs(mean_diff))) %>%  # 按效应大小排序
      filter(consistency_score == 4) %>%
      head(n_top) 
  
  if (nrow(top_data) == 0) {
    cat("没有足够的数据可绘制环形图\n")
    return(NULL)
  }
  
  # 获取所有细胞类型
  all_cells <- unique(c(top_data$source, top_data$target))
  
  # 创建PDF
  circos_file <- file.path(outputdir, paste0("directional_circos_", comp_name, ".pdf"))
  pdf(circos_file, width = 12, height = 12)
  
  # 初始化circos
  circos.clear()
  circos.par("gap.degree" = 3, "start.degree" = 90, 
             "track.height" = 0.08, "cell.padding" = c(0, 0, 0, 0))
  
  # 初始化扇形
  circos.initialize(factors = factor(all_cells, levels = all_cells), 
                    xlim = c(0, 1))
  
  # 绘制细胞类型轨道
  circos.trackPlotRegion(
    ylim = c(0, 1),
    track.height = 0.1,
    bg.border = "gray70",
    bg.col = "white",
    panel.fun = function(x, y) {
      sector_index <- CELL_META$sector.index
      circos.text(
        x = 0.5,
        y = 0.5,
        labels = sector_index,
        sector.index = sector_index,
        facing = "clockwise",
        niceFacing = TRUE,
        adj = c(0.5, 0.5),
        cex = 0.9,
        col = "black",
        font = 2
      )
    }
  )
  
  # 绘制带箭头的连线
  for (i in 1:nrow(top_data)) {
    source_cell <- top_data$source[i]
    target_cell <- top_data$target[i]
    mean_diff <- top_data$mean_diff[i]
    
    # 设置颜色和宽度
    color <- ifelse(mean_diff > 0, "#E76F51", "#2A9D8F")
    width <- max(1, min(abs(mean_diff) * 100, 8))
    
    # 计算串联位置
    source_y <- 0.7 + (0.2 * (i-1) / nrow(top_data))
    target_y <- 0.1 + (0.2 * (i-1) / nrow(top_data))
    
    # 绘制带箭头的连线
    circos.link(
      sector.index1 = source_cell,
      point1 = source_y,
      sector.index2 = target_cell,
      point2 = target_y,
      col = color,
      lwd = width,
      border = NA,
      arr.type = "triangle",
      arr.length = 1,
      arr.width = 1,
      arr.lwd = 1,
      arr.col = color,
      directional = 1
    )
  }
  
  # 添加图例和标题
  par(new = TRUE)
  plot(0, 0, type = "n", xlim = c(0, 1), ylim = c(0, 1), 
       axes = FALSE, xlab = "", ylab = "")
  
  legend("bottomleft",
         legend = c("Source → Target", 
                   "Salmon: A > B (上调)", 
                   "Olive: A < B (下调)"),
         col = c("black", "#E76F51", "#2A9D8F"),
         lty = c(NA, 1, 1),
         lwd = c(NA, 3, 3),
         pch = c(NA, NA, NA),
         bty = "n", 
         cex = 0.8)
  
  title(main = paste("Directed Cell Communication Network:", comp_name),
        sub = paste("Top", n_top, "interactions | Arrows show direction"),
        cex.main = 1.2, cex.sub = 0.8)
  
  circos.clear()
  dev.off()
  
  cat("带箭头的环形图已保存至:", circos_file, "\n")
  return(top_data)
}

# ============================================================================
# 6. 保守可视化函数
# ============================================================================

generate_conservative_plots <- function(statistical_results, comp_name, outputdir) {
  
  # 创建输出目录
  if (!dir.exists(outputdir)) {
    dir.create(outputdir, recursive = TRUE)
  }
  
  # 1. 一致性分析图
  p1 <- ggplot(statistical_results, aes(x = consistency_score, y = abs(mean_diff))) +
    geom_point(aes(color = priority_level, 
                   size = ifelse(is.na(p_adjusted), 1, -log10(p_adjusted + 1e-10))), 
               alpha = 0.7) +
    scale_color_manual(values = c("high" = "#E76F51", 
                                 "medium_high" = "#F4A261", 
                                 "medium" = "#2A9D8F", 
                                 "low" = "gray")) +
    labs(title = paste("Consistency vs Effect Size:", comp_name),
         x = "Consistency Score (1-4)",
         y = "Absolute Mean Difference",
         color = "Priority Level",
         size = "-log10(p)") +
    theme_minimal()
  
  ggsave(file.path(outputdir, paste0("consistency_plot_", comp_name, ".pdf")), 
         p1, width = 10, height = 8)
  
  # 2. 方向一致性分布
  dir_data <- statistical_results %>%
    count(dominant_direction) %>%
    mutate(prop = n / sum(n),
           dominant_direction = factor(dominant_direction,
                                       levels = c("all_up", "mostly_up", 
                                                 "inconsistent", 
                                                 "mostly_down", "all_down")))
  
  p2 <- ggplot(dir_data, aes(x = dominant_direction, y = prop, 
                             fill = dominant_direction)) +
    geom_col() +
    geom_text(aes(label = paste0(n, "\n(", round(prop*100, 1), "%)")), 
              vjust = -0.3, size = 3) +
    scale_fill_manual(values = c("all_up" = "#E76F51", 
                                "mostly_up" = "#F4A261",
                                "inconsistent" = "gray70",
                                "mostly_down" = "#8AB17D", 
                                "all_down" = "#2A9D8F")) +
    labs(title = paste("Direction Consistency Distribution:", comp_name),
         x = "Dominant Direction",
         y = "Proportion") +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))
  
  ggsave(file.path(outputdir, paste0("direction_distribution_", comp_name, ".pdf")), 
         p2, width = 8, height = 6)
  
  # 3. Top 20 交互条形图
   top_data <- statistical_results %>%
      filter(p_value_raw < 0.05) %>%  # 首先筛选显著的结果
      arrange(desc(abs(mean_diff))) %>%  # 按效应大小排序
      filter(consistency_score == 4) %>%
      head(20) 
  
  if (nrow(top_data) > 0) {
    top_data$interaction <- paste(top_data$source, "→", top_data$target)
    top_data$interaction <- factor(top_data$interaction, 
                                  levels = top_data$interaction[order(top_data$mean_diff)])
    
    p3 <- ggplot(top_data, aes(x = interaction, y = mean_diff)) +
      geom_bar(aes(fill = ifelse(mean_diff > 0, "#E76F51", "#2A9D8F")), 
               stat = "identity", width = 0.7) +
      scale_fill_identity() +
      coord_flip() +
      labs(title = paste("Top 20 Largest Changes:", comp_name),
           x = "Cell-Cell Interaction",
           y = "Mean Difference (A - B)") +
      theme_minimal() +
      theme(axis.text.y = element_text(size = 9))
    
    ggsave(file.path(outputdir, paste0("top20_changes_", comp_name, ".pdf")), 
           p3, width = 10, height = 8)
  }
  
  # 4. 生成环形图
  circos_data <- create_directional_circos(statistical_results, comp_name, outputdir, n_top = 20)
  
  cat("\n所有可视化已保存至:", outputdir, "\n")
  cat("生成的图形：\n")
  cat("1. consistency_plot_", comp_name, ".pdf\n", sep = "")
  cat("2. direction_distribution_", comp_name, ".pdf\n", sep = "")
  cat("3. top20_changes_", comp_name, ".pdf\n", sep = "")
  cat("4. directional_circos_", comp_name, ".pdf\n", sep = "")
  
  return(list(
    consistency_plot = p1,
    direction_plot = p2,
    top_changes_plot = if (exists("p3")) p3 else NULL,
    circos_data = circos_data
  ))
}

# ============================================================================
# 7. 完整工作流程函数
# ============================================================================

run_complete_analysis_2x2 <- function(cellchat_objects, sample_names, 
                                      group_A, group_B, target_types, 
                                      comp_name, outputdir) {
  
  # 打印标题
  separator_line <- paste(rep("=", 60), collapse = "")
  
  cat(separator_line, "\n")
  cat("开始2×2分析:", comp_name, "\n")
  cat("Group A:", paste(group_A, collapse = ", "), "\n")
  cat("Group B:", paste(group_B, collapse = ", "), "\n")
  cat("目标细胞类型:", length(target_types), "种\n")
  cat(separator_line, "\n\n")
  
  # 步骤1: 提取通讯矩阵
  cat("步骤1: 提取通讯矩阵...\n")
  all_matrices <- list()
  
  for (sample_name in sample_names) {
    cat("处理样本:", sample_name, "...")
    
    if (!sample_name %in% names(cellchat_objects)) {
      cat("错误: 未找到样本", sample_name, "的CellChat对象\n")
      next
    }
    
    matrix_df <- extract_communication_matrix(
      cellchat_objects[[sample_name]], 
      sample_name, 
      target_types
    )
    
    all_matrices[[sample_name]] <- matrix_df
    cat("完成\n")
  }
  
  combined_data <- bind_rows(all_matrices)
  
  # 步骤2: 执行所有配对比较
  cat("\n步骤2: 执行配对比较...\n")
  all_comparisons <- list()
  pair_counter <- 1
  
  for (a_sample in group_A) {
    for (b_sample in group_B) {
      cat("比较:", a_sample, "vs", b_sample, "...")
      
      comparison_result <- compare_two_samples(combined_data, a_sample, b_sample)
      comparison_result$pair_id <- pair_counter
      comparison_result$comparison_name <- paste(a_sample, "vs", b_sample)
      
      all_comparisons[[as.character(pair_counter)]] <- comparison_result
      pair_counter <- pair_counter + 1
      
      cat("完成\n")
    }
  }
  
  all_pairwise_results <- bind_rows(all_comparisons)
  
  # 步骤3: 统计分析
  cat("\n步骤3: 统计分析...\n")
  statistical_results <- perform_statistical_analysis_2x2(
    all_pairwise_results, 
    group_A, 
    group_B
  )
  
  # 步骤4: 网络可视化
  cat("\n步骤4: 网络可视化...\n")
  network_result <- plot_cell_network_2x2(statistical_results, comp_name, outputdir)
  
  # 步骤5: 生成保守可视化
  cat("\n步骤5: 生成可视化...\n")
  visualization_results <- generate_conservative_plots(
    statistical_results, 
    comp_name, 
    outputdir
  )
  
  cat("\n", separator_line, "\n")
  cat("分析完成! 结果已保存至:", outputdir, "\n")
  cat("注意: 2×2设计统计功效有限，请谨慎解释结果\n")
  cat(separator_line, "\n")
  
  # 返回所有结果
  return(list(
    combined_data = combined_data,
    all_comparisons = all_pairwise_results,
    statistical_results = statistical_results,
    network_result = network_result,
    visualization_results = visualization_results,
    summary = list(
      total_interactions = nrow(statistical_results),
      high_consistency = sum(statistical_results$consistency_score == 4),
      significant_05 = sum(statistical_results$p_value_raw < 0.05, na.rm = TRUE),
      significant_10 = sum(statistical_results$p_value_raw < 0.1, na.rm = TRUE)
    )
  ))
}


# ============================================================================
# 8. 批量分析多个比较
# ============================================================================

run_batch_analyses <- function(cellchat_objects, comparisons_list, target_types, output_base_dir) {
  results <- list()
  
  for (i in seq_along(comparisons_list)) {
    comp <- comparisons_list[[i]]
    comp_name <- comp$name
    group_A <- comp$group_A
    group_B <- comp$group_B
    
    cat("\n开始分析:", comp_name, "\n")
    cat("Group A:", paste(group_A, collapse = ", "), "\n")
    cat("Group B:", paste(group_B, collapse = ", "), "\n")
    
    # 创建输出目录
    outputdir <- file.path(output_base_dir, comp_name)
    
    # 运行分析
    result <- tryCatch({
      run_complete_analysis_2x2(
        cellchat_objects = cellchat_objects,
        sample_names = unique(c(group_A, group_B)),
        group_A = group_A,
        group_B = group_B,
        target_types = target_types,
        comp_name = comp_name,
        outputdir = outputdir
      )
    }, error = function(e) {
      cat("分析失败:", e$message, "\n")
      return(NULL)
    })
    
    if (!is.null(result)) {
      results[[comp_name]] <- result
    }
  }
  
  return(results)
}

# 打印加载信息
cat("CellChat 2×2 设计分析工具已加载\n")
cat("版本: 2.0 (修复版)\n")
cat("主要功能:\n")
cat("1. 2×2设计的细胞通讯分析\n")
cat("2. 小样本统计分析\n")
cat("3. 多种可视化输出\n")
cat("4. 完整工作流程\n\n")
cat("输入 example_usage() 查看使用示例\n")
cat("输入 check_data_compatibility() 检查数据兼容性\n")

In [ ]:
target_celltypes <- c(#'CD4.exh', 'CD4.naive', 'CD4.reg',
                      'CD8.cyto', 'CD8.naive', 'CD8.prolif', 'CD8.RM',
                      #'ID8', 'ID8_ADV', 'ID8_apop', 'ID8_apop_ADV', 'ID8_prolif', 'ID8_prolif_ADV', 
                      'TAM.Ccl', 'TAM.Ccr2', 'TAM.Fth1', 'TAM.M2',
                     'TAM.plac8', 'TAM.prolif')

In [ ]:
# 运行分析
comp_name =  "oAd-exCD40L_vs_oAd-wtCD40L_2x2_CD42Macro"
results <- run_complete_analysis_2x2(
  cellchat_objects = cellchats,  # 您的CellChat对象列表
  sample_names =  c('M20_5_V21','M20_5_V47','M20_3_V4','M20_3_V5'),
  group_A = c('M20_5_V21','M20_5_V47'),  # 对照组，2个样本
  group_B = c('M20_3_V4','M20_3_V5'),  # 处理组，2个样本
 target_types = c('CD4.exh', 'CD4.naive', 'CD4.reg',
                      #'CD8.cyto', 'CD8.naive', 'CD8.prolif', 'CD8.RM',
                      #'ID8', 'ID8_ADV', 'ID8_apop', 'ID8_apop_ADV', 'ID8_prolif', 'ID8_prolif_ADV', 
                      'TAM.Ccl', 'TAM.Ccr2', 'TAM.Fth1', 'TAM.M2',
                    'TAM.plac8', 'TAM.prolif'),
  comp_name = comp_name,
  outputdir = file.path(outputdir, comp_name)
)



In [ ]:
# 2. 加载你的CellChat对象
# cellchat_objects <- list(...)
comp_name =  "oAd-exCD40L_vs_oAd_2x2_CD42Macro"
results <- run_complete_analysis_2x2(
  cellchat_objects = cellchats,  # 您的CellChat对象列表
  sample_names =  c('M20_5_V21','M20_5_V47','M20_V40','M20_V42'),# put test group at tail
  group_A = c('M20_5_V21','M20_5_V47'),  # 对照组，2个样本
  group_B = c('M20_V40','M20_V42'),  # 处理组，2个样本
 target_types = c('CD4.exh', 'CD4.naive', 'CD4.reg',
                      #'CD8.cyto', 'CD8.naive', 'CD8.prolif', 'CD8.RM',
                      #'ID8', 'ID8_ADV', 'ID8_apop', 'ID8_apop_ADV', 'ID8_prolif', 'ID8_prolif_ADV', 
                      'TAM.Ccl', 'TAM.Ccr2', 'TAM.Fth1', 'TAM.M2',
                     'TAM.plac8', 'TAM.prolif'),
  comp_name = comp_name,
  outputdir = file.path(outputdir, comp_name)
)



In [ ]:
# 运行分析
comp_name =  "oAd-wtCD40L_vs_oAd_2x2_CD42Macro"
results <- run_complete_analysis_2x2(
  cellchat_objects = cellchats,  # 您的CellChat对象列表
  sample_names =  c('M20_3_V4','M20_3_V5','M20_V40','M20_V42'),# put test group at tail
  group_A = c('M20_3_V4','M20_3_V5'),  # 对照组，2个样本
  group_B = c('M20_V40','M20_V42'),  # 处理组，2个样本
 target_types = c('CD4.exh', 'CD4.naive', 'CD4.reg',
                     # 'CD8.cyto', 'CD8.naive', 'CD8.prolif', 'CD8.RM',
                      #'ID8', 'ID8_ADV', 'ID8_apop', 'ID8_apop_ADV', 'ID8_prolif', 'ID8_prolif_ADV', 
                      'TAM.Ccl', 'TAM.Ccr2', 'TAM.Fth1', 'TAM.M2',
                     'TAM.plac8', 'TAM.prolif'),
  comp_name = comp_name,
  outputdir = file.path(outputdir, comp_name)
)



In [ ]:
# 细胞亚群互作网络图
plot_cell_network <- function(lr_result, comp_name) {
  # 汇总每个细胞对的总体变化
  cell_pair_summary <- lr_result %>%
    group_by(source, target) %>%
    summarise(
      total_change = sum(abs(difference)),
      mean_change = mean(difference),
      n_lr_pairs = n(),
      .groups = "drop"
    )
  
  # 创建网络图
  library(igraph)
  
  # 创建图对象
  g <- graph_from_data_frame(cell_pair_summary, directed = TRUE)
  
  # 设置边的属性
  E(g)$weight <- cell_pair_summary$total_change
  E(g)$color <- ifelse(cell_pair_summary$mean_change > 0, "purple", "seagreen")
  E(g)$width <- scales::rescale(cell_pair_summary$total_change, to = c(1, 8))
  
  # 设置节点的属性（细胞类型）
  V(g)$size <- 15
  V(g)$label.cex <- 0.8
  
  # 绘制网络图
  pdf(paste0(outputdir,"network_", comp_name, ".pdf"), width = 12, height = 10)
  plot(g, 
       layout = layout_with_fr(g),
       edge.arrow.size = 0.5,
       edge.arrow.width = 1,
       main = paste("network:", comp_name))
  legend("bottomleft", 
         legend = c("up", "dn"), 
         col = c("purple", "seagreen"), 
         lwd = 2, 
         bty = "n")
  dev.off()
}

In [ ]:
comparison_outputdir <- file.path(outputdir, "CD8_Macro_cells_comp")
dir.create(comparison_outputdir, recursive = TRUE, showWarnings = FALSE)


In [ ]:
# 2. 定义要分析的细胞亚群
target_celltypes <- c(#'CD4.exh', 'CD4.naive', 'CD4.reg',
                      'CD8.cyto', 'CD8.naive', 'CD8.prolif', 'CD8.RM',
                      #'ID8', 'ID8_ADV', 'ID8_apop', 'ID8_apop_ADV', 'ID8_prolif', 'ID8_prolif_ADV', 
                      'TAM.Ccl', 'TAM.Ccr2', 'TAM.Fth1', 'TAM.M2',
                     'TAM.plac8', 'TAM.prolif')

In [ ]:


dir.create(comparison_outputdir, showWarnings = TRUE, recursive = FALSE)
# 4. 为所有样本提取communication数据
all_comm_data <- list()
for (i in 1:length(cellchats)) {
  sample_name <- names(cellchats)[i]
  cat("\n正在处理样本:", sample_name, "\n")
  all_comm_data[[sample_name]] <- extract_communication_matrix(cellchats[[i]], sample_name, target_celltypes)
}

# 合并所有数据
comm_df <- bind_rows(all_comm_data)
# 7. 执行所有样本对的比较
comparison_results <- list()

for (pair in sample_pairs) {
  sample1 <- pair[1]
  sample2 <- pair[2]
  
  cat("\n==================================================\n")
  cat("比较:", sample1, "vs", sample2, "\n")
  cat("==================================================\n")
  
  result <- compare_two_samples(comm_df, sample1, sample2)
  comparison_name <- paste0(sample1, "_vs_", sample2)
  comparison_results[[comparison_name]] <- result
  
  # 统计信息
  n_up <- sum(result$change_direction == "upregulated")
  n_down <- sum(result$change_direction == "downregulated")
  n_total <- nrow(result)
  
  cat("总共有", n_total, "个细胞亚群对在两个样本中都存在\n")
  cat("上调的communication:", n_up, "\n")
  cat("下调的communication:", n_down, "\n")
  cat("不变的communication:", n_total - n_up - n_down, "\n")
  
  # 保存结果
  write.csv(result, file.path(comparison_outputdir, paste0("communication_changes_", comparison_name, ".csv")), row.names = FALSE)
}

# 8. 生成汇总报告
cat("\n\n最终汇总报告:\n")
cat("==================================================\n")

for (comp_name in names(comparison_results)) {
  result <- comparison_results[[comp_name]]
  
  # 找出变化最显著的interaction
  top_up <- result %>% 
    filter(change_direction == "upregulated") %>% 
    arrange(desc(strength_diff)) %>% 
    head(5)
  
  top_down <- result %>% 
    filter(change_direction == "downregulated") %>% 
    arrange(strength_diff) %>% 
    head(5)
  
  cat("\n比较:", comp_name, "\n")
  
  if (nrow(top_up) > 0) {
    cat("上调最显著的interactions:\n")
    for (i in 1:nrow(top_up)) {
      cat("  ", top_up$source[i], "->", top_up$target[i], 
          "(差异:", round(top_up$strength_diff[i], 4), ")\n")
    }
  }
  
  if (nrow(top_down) > 0) {
    cat("下调最显著的interactions:\n")
    for (i in 1:nrow(top_down)) {
      cat("  ", top_down$source[i], "->", top_down$target[i], 
          "(差异:", round(top_down$strength_diff[i], 4), ")\n")
    }
  }
  cat("---\n")
}


In [ ]:

# 为每个比较生成图表
for (comp_name in names(comparison_results)) {
  p <- generate_comparison_plot(comparison_results[[comp_name]], comp_name)
  if (!is.null(p)) {
    plot_file <- file.path(comparison_outputdir, paste0("visualization_", comp_name, "_CD8_Macro.pdf"))
    ggsave(plot_file, p, width = 10, height = 8)
    cat("已生成可视化图表:", plot_file, "\n")
  }
}

cat("\n分析完成！所有结果已保存到CSV文件和PDF图表中。\n")